In [304]:
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import torch.nn.functional as F
import torch.nn as nn


In [288]:
df = pd.read_csv("C:\\Users\\j\\Desktop\\pytorch-tutorials\\data.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [289]:
df.shape

(569, 33)

In [290]:
df.drop(columns = ["id", "Unnamed: 32"], inplace = True)

In [291]:
x = df.drop(columns = ["diagnosis"])
y = df["diagnosis"]

In [292]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [293]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [294]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

## converting numpy into tensors

In [295]:
x_train_tensor = torch.from_numpy(x_train).float()
x_test_tensor = torch.from_numpy(x_test).float()

y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)

## Defining model

In [296]:
class MyANNmodel():

    def __init__(self, x):

        self.weights = torch.rand(x.shape[1], 1, dtype = torch.float32, requires_grad = True)
        self.bias = torch.zeros(1, dtype = torch.float32, requires_grad = True)
        
    def forwardpass(self,x):
        z = torch.matmul(x, self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred

In [297]:
learning_rate = 0.1
epochs = 5

In [298]:
model = MyANNmodel(x_train_tensor)

In [299]:
model.weights

tensor([[0.9169],
        [0.0452],
        [0.0170],
        [0.8665],
        [0.0538],
        [0.5638],
        [0.2077],
        [0.9412],
        [0.0258],
        [0.9099],
        [0.7556],
        [0.0126],
        [0.9055],
        [0.1553],
        [0.1683],
        [0.8049],
        [0.9439],
        [0.5190],
        [0.0123],
        [0.3759],
        [0.6102],
        [0.6521],
        [0.4671],
        [0.0341],
        [0.0670],
        [0.1781],
        [0.1187],
        [0.0737],
        [0.2402],
        [0.2148]], requires_grad=True)

## Training model

In [ ]:
# creating model


# training loop
for epoch in range(epochs):
    
    # creating forward pass
    y_pred = model.forwardpass(x_train_tensor)
    
    # loss function
    
    loss = F.binary_cross_entropy(y_pred, y_train_tensor, reduction = "mean"   )
    
    
    # backward pass
    loss.backward()
    
    # parameter optimization
    with torch.no_grad(): # to off gradient tracking 
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad
        
    # zero gradients
    model.weights.grad.zero_()
    model.bias.grad.zero_()
    
    # print loss
    print(f"epoch: {epoch + 1} loss: {loss.item():.4f}")

epoch: 1 loss: 0.6836
epoch: 2 loss: 0.6690
epoch: 3 loss: 0.6549
epoch: 4 loss: 0.6412
epoch: 5 loss: 0.6279


In [301]:
model.weights

tensor([[ 0.9445],
        [ 0.0623],
        [ 0.0414],
        [ 0.8911],
        [ 0.0409],
        [ 0.5273],
        [ 0.1892],
        [ 0.9423],
        [ 0.0066],
        [ 0.8455],
        [ 0.7581],
        [-0.0067],
        [ 0.9014],
        [ 0.1636],
        [ 0.1264],
        [ 0.7315],
        [ 0.8881],
        [ 0.4696],
        [-0.0179],
        [ 0.2985],
        [ 0.6402],
        [ 0.6795],
        [ 0.4926],
        [ 0.0598],
        [ 0.0705],
        [ 0.1565],
        [ 0.1038],
        [ 0.0788],
        [ 0.2548],
        [ 0.1739]], requires_grad=True)

## Evaluation

In [302]:
y_pred = model.forwardpass(x_test_tensor)

In [303]:
y_pred = (y_pred > 0.9).float()
accuracy = (y_pred == y_test_tensor).float().mean()
print(f"accuracy: {accuracy:.4f}")

accuracy: 0.9123


# Training Neural Network using NN module

In [ ]:
class MyModule(nn.Module):
    
    def __init__(self, features):
        
        super().__init__()
        self.linear = nn.Linear(features, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, features):
        out = self.linear(features)
        out = self.sigmoid(out)
        return out
    
    def loss(self, y_pred, y_true):
        return F.binary_cross_entropy(y_pred, y_true, reduction = "mean")
        

In [ ]:
model = MyModule(x_train_tensor.shape[1])

for epoch in range(epochs):
    
    y_pred = model(x_train_tensor)
    
    loss = model.loss(y_pred, y_train_tensor)
    
    loss.backward()
    
    with torch.no_grad():
        model.linear.weight -= learning_rate * model.linear.weight.grad
        model.linear.bias -= learning_rate * model.linear.bias.grad
    
    model.linear.weight.grad.zero_()
    model.linear.bias.grad.zero_()
    
    print(f"epoch: {epoch +1} loss: {loss.item():.4f}")

AttributeError: module 'torch.nn' has no attribute 'linear'